# 1 - Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [3]:
from src.utils import config, io

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


# 2 - Data Extraction

## 2.1 - OECD Rating

In [108]:
from src.extraction import build_oecd_dataset

In [113]:
oecd_rating_matrix_previous = build_oecd_dataset.get_clean_oecd_rating_df(oecd_fname='02-02-2024') # 02-02-2024
oecd_rating_matrix_previous

Reading CSV


,1999-01-01,2000-01-01,2001-01-01,2002-01-01,2003-01-01,2004-01-01,2005-01-01,2006-01-01,2007-01-01,2008-01-01,...,2015-01-01,2016-01-01,2017-01-01,2018-01-01,2019-01-01,2020-01-01,2021-01-01,2022-01-01,2023-01-01,2024-01-01
ISO3_COUNTRY_CODE,,,,,,,,,,,,,,,,,,,,,
AFG,7,7,-,-,-,-,-,-,-,7,...,7,7,7,7,7,7,7,7,7,7
ALB,7,7,7,7,7,7,6,6,6,6,...,6,6,6,6,6,5,5,5,5,5
DZA,6,6,5,5,4,4,4,3,3,3,...,3,3,4,4,4,4,5,5,5,5
AND,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
AGO,7,7,7,7,7,7,7,7,7,7,...,5,5,6,6,6,6,6,6,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VNM,6,6,6,6,5,5,5,5,5,4,...,5,5,5,5,5,4,4,4,4,4
PSE,-,-,-,-,-,-,-,-,-,-,...,7,7,7,7,7,7,7,7,7,7
YEM,7,7,7,6,6,6,6,6,6,6,...,7,7,7,7,7,7,7,7,7,7


In [115]:
import numpy as np
oecd_rating_matrix_previous.replace('-', np.nan).isna().sum().sum()

np.int64(850)

In [ ]:
oecd_rating_matrix = build_oecd_dataset.get_clean_oecd_rating_df(oecd_fname='03-07-2026') # 02-02-2024
oecd_rating_matrix

,1999-01-01,2000-01-01,2001-01-01,2002-01-01,2003-01-01,2004-01-01,2005-01-01,2006-01-01,2007-01-01,2008-01-01,...,2017-01-01,2018-01-01,2019-01-01,2020-01-01,2021-01-01,2022-01-01,2023-01-01,2024-01-01,2025-01-01,2026-01-01
ISO3_COUNTRY_CODE,,,,,,,,,,,,,,,,,,,,,
AFG,7,7,-,-,-,-,-,-,-,7,...,7,7,7,7,7,7,7,7,7,7
ALB,7,7,7,7,7,7,6,6,6,6,...,6,6,6,5,5,5,5,5,5,4
DZA,6,6,5,5,4,4,4,3,3,3,...,4,4,4,5,5,5,5,5,5,5
AND,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
AGO,7,7,7,7,7,7,7,7,7,7,...,6,6,6,6,6,6,6,6,6,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
VNM,6,6,6,6,5,5,5,5,5,4,...,5,5,5,4,4,4,4,4,4,4
PSE,-,-,-,-,-,-,-,-,-,-,...,7,7,7,7,7,7,7,7,7,7
YEM,7,7,7,6,6,6,6,6,6,6,...,7,7,7,7,7,7,7,7,7,7


In [117]:
import numpy as np
oecd_rating_matrix.replace('-', np.nan).isna().sum().sum()

np.int64(947)

In [119]:
import numpy as np
oecd_rating_matrix.replace('-', np.nan).isna().sum().sum()

np.int64(909)

In [109]:
io.save_csv(oecd_rating_matrix, config.INTERIM_DATA_DIR / 'oecd_rating_matrix.csv', index=True)

## 2.2 - World Bank Features

In [110]:
from src.extraction import world_bank, world_bank_WDI_indicators

In [111]:
oecd_rating_matrix = io.load_csv(config.INTERIM_DATA_DIR / 'oecd_rating_matrix.csv', index_col=0)
oecd_countries = list(oecd_rating_matrix.index)
min_year = int(min(oecd_rating_matrix.columns)[:4])
max_year = int(max(oecd_rating_matrix.columns)[:4])

In [149]:
wb_df = world_bank.get_world_bank_indicators(
    indicators=world_bank_WDI_indicators.WORLD_BANK_INDICATORS,
    countries=oecd_countries,
    start_year=min_year,
    end_year=max_year,
)
wb_df

Capping end year to 2025 (WB data not yet available beyond that)
[1/74] GDP (current US$) (NY.GDP.MKTP.CD) (fetching…)
[2/74] GDP growth (annual %) (NY.GDP.MKTP.KD.ZG) (fetching…)
[3/74] GDP, PPP (current international $) (NY.GDP.MKTP.PP.CD) (fetching…)
[4/74] GDP per capita (current US$) (NY.GDP.PCAP.CD) (fetching…)
[5/74] GDP per capita growth (annual %) (NY.GDP.PCAP.KD.ZG) (fetching…)
[6/74] Gross capital formation (annual % growth) (NE.GDI.TOTL.KD.ZG) (fetching…)
[7/74] Gross capital formation (% of GDP) (NE.GDI.TOTL.ZS) (fetching…)
[8/74] Industry (including construction), value added (current US$) (NV.IND.TOTL.CD) (fetching…)
[9/74] Industry (including construction), value added (annual % growth) (NV.IND.TOTL.KD.ZG) (fetching…)
[10/74] Industry (including construction), value added (% of GDP) (NV.IND.TOTL.ZS) (fetching…)
[11/74] Employment in industry (% of total employment) (modeled ILO estimate) (SL.IND.EMPL.ZS) (fetching…)
[12/74] Services, value added (current US$) (NV.SRV.TO

,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,NY.GDP.PCAP.CD,NY.GDP.PCAP.KD.ZG,NE.GDI.TOTL.KD.ZG,NE.GDI.TOTL.ZS,NV.IND.TOTL.CD,NV.IND.TOTL.KD.ZG,NV.IND.TOTL.ZS,...,SP.POP.GROW,SP.URB.TOTL.IN.ZS,EN.POP.DNST,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,IQ.CPA.BREG.XQ,IQ.CPA.PADM.XQ,FS.AST.PRVT.GD.ZS,FM.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
ABW-1999,1.722905e+09,1.238042,2.489441e+09,19216.197235,-0.125966,NaN,30.869001,2.678827e+08,NaN,15.548314,...,1.356486,65.361486,498.105556,NaN,43.669781,NaN,NaN,NaN,44.620525,44.620525
AFG-1999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.728116,18.390327,30.491981,NaN,108.686031,46.609,NaN,NaN,NaN,NaN
AGO-1999,6.152923e+09,2.181490,5.669622e+10,392.725539,-1.130799,NaN,NaN,NaN,NaN,NaN,...,3.295277,49.259527,12.566965,NaN,93.586677,77.380,NaN,NaN,NaN,2.573386
ALB-1999,3.283942e+09,12.250728,1.103390e+10,1056.344812,12.963927,31.19083,24.271511,6.082915e+08,20.139252,18.523214,...,-0.633352,40.897284,113.459051,NaN,59.064263,62.061,NaN,NaN,NaN,4.116988
AND-1999,1.240295e+09,4.099079,2.075776e+09,18875.288370,3.579455,NaN,NaN,NaN,NaN,NaN,...,0.500413,92.540282,139.808511,NaN,37.399632,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WSM-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,42.534,NaN,NaN,NaN,NaN
YEM-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,33.103,NaN,NaN,NaN,NaN
ZAF-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,55.584,NaN,NaN,NaN,NaN


In [151]:
io.save_csv(wb_df, config.INTERIM_DATA_DIR / 'worldbank_dataset_extracted.csv', index=True)